# market_neural_net — Colab training launcher

Reusable GPU launcher shell per the project README (§2.1: local CPU handles data/backtesting, this notebook handles GPU-hungry training). Right now that means the M2 LSTM baseline (which is actually CPU-viable and just being run here to verify the whole pipeline works end to end on Colab); this is the notebook that will later run Phase 3's transformer pretraining, which genuinely needs the GPU.

**Before running:** upload two files to your Google Drive, produced by `python -m scripts.package_for_colab` on your own machine:
- `My Drive/market_neural_net/colab_code_bundle.zip` (src/, scripts/, configs/, requirements.txt — small, re-upload whenever code changes)
- `My Drive/market_neural_net/colab_data_bundle.zip` (the curated dataset, ~300MB — upload once, re-upload only after a real re-ingest)

There's no GitHub remote for this project yet, so this notebook unzips from Drive instead of `git clone`. Once a remote exists, swap Cell 3 for a `git clone` + `pip install -e .` and delete the packaging step — everything downstream is unaffected.

**Known gap, read before a long run:** the training script here does NOT checkpoint/resume mid-run yet. That's fine for the LSTM baseline (minutes, not hours) but MUST be added before running Phase 3's transformer pretraining here — Colab sessions disconnect, and losing a multi-hour run to that is exactly the failure mode §2.1 warns about. Don't skip that step later just because this notebook runs today.

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU assigned — Runtime > Change runtime type > GPU, then re-run this cell.')

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/My Drive/market_neural_net'  # change if you uploaded elsewhere

## 3. Unpack code + data bundles into the Colab VM

In [ ]:
import zipfile, os

PROJECT_DIR = '/content/market_neural_net'
os.makedirs(PROJECT_DIR, exist_ok=True)

code_zip = f'{DRIVE_PROJECT_DIR}/colab_code_bundle.zip'
data_zip = f'{DRIVE_PROJECT_DIR}/colab_data_bundle.zip'

assert os.path.exists(code_zip), f'missing {code_zip} — upload it from your machine first'
with zipfile.ZipFile(code_zip) as zf:
    zf.extractall(PROJECT_DIR)
print('code extracted')

if os.path.exists(data_zip):
    with zipfile.ZipFile(data_zip) as zf:
        zf.extractall(PROJECT_DIR)
    print('data extracted')
else:
    print(f'WARNING: {data_zip} not found — training will fail without curated data')

%cd $PROJECT_DIR

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Sanity checks: tests, then a tiny smoke-test training run
Per README §2.1 rule 1 — always verify the tiny config before trusting a real run on a new machine.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
# Tiny config smoke test — should finish in well under a minute even on CPU.
!python -m scripts.train_lstm_baseline \
  --n-symbols 5 --seq-len 20 --hidden-size 16 --num-layers 1 --epochs 1 \
  --test-years 2024 --out-dir experiments/lstm_baseline_smoketest

## 6. Real training run
Full README-spec baseline: 40 liquid symbols, seq_len=120, hidden=128, 2-layer LSTM, 4 walk-forward folds. On a T4 this should be well under the local-CPU runtime. Edit the flags here directly for a different sweep, or point `--curated-dir`/change the script entirely once Phase 3's transformer is ready.

In [ ]:
!python -m scripts.train_lstm_baseline \
  --n-symbols 40 --seq-len 120 --hidden-size 128 --num-layers 2 --epochs 6 \
  --test-years 2022 2023 2024 2025 \
  --out-dir experiments/lstm_baseline

## 7. Sync results back to Drive
So the report/checkpoints survive the Colab VM being recycled.

In [ ]:
import shutil
dest = f'{DRIVE_PROJECT_DIR}/experiments_from_colab'
shutil.copytree('experiments', dest, dirs_exist_ok=True)
print('synced experiments/ ->', dest)